# 第3章：MPI SpMV

> 实验主入口：[MPI SpMV 实验手册](EXPERIMENT_GUIDE.md)。手册区分单机多进程与多节点实验，并提供直接 `mpirun` 命令和记录表。

OpenMP 线程共享同一地址空间；MPI 进程拥有独立地址空间，必须显式分发 CSR、广播 x 并汇聚 y。本章使用原 mpi-SpMV 的真实实现学习这一转换。

## 前置要求

- 完成 OpenMP SpMV 章节
- 理解 CSR 和加速比
- 目标环境提供 mpicxx 与 mpirun

## 本章学习目标

- 说明 rank/size/communicator
- 实现按 nnz 平衡的 CSR 分区
- 分解通信、局部计算和汇聚耗时

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import shutil, subprocess
for tool in ("cmake", "mpicxx", "mpirun"):
    path = shutil.which(tool)
    if path is None:
        raise RuntimeError(f"缺少必需工具：{tool}")
    print(f"{tool}: {path}")


## 章节内容

- [03.01_chapter_intro](03.01_chapter_intro.ipynb)：从 OpenMP 过渡到 MPI
- [03.02_mpi_programming_model](03.02_mpi_programming_model.ipynb)：进程、rank 和 communicator
- [03.03_csr_data_partitioning](03.03_csr_data_partitioning.ipynb)：按累计 nnz 分区
- [03.04_distributed_spmv_implementation](03.04_distributed_spmv_implementation.ipynb)：集合通信数据流
- [03.05_mpi_scaling_analysis](03.05_mpi_scaling_analysis.ipynb)：通信计算与扩展性
- [03.06_chapter_test](03.06_chapter_test.ipynb)：进程数单变量实验

## 实验材料说明

本章完整工程位于当前章节的 `src/`，练习参考位于 `answer/`。Notebook 使用相对路径访问材料，不依赖开发者本机的原始工程位置。

## 预期现象与结果分析

上面的目录检查应显示本章 Notebook、`answer`、`images` 和 `src`。若文件缺失，应先检查课程检出是否完整，而不是继续执行后续实验。按目录顺序学习，先确认正确性，再记录性能；不要用未启用真实后端的 stub 或 reference 路径代表 NPU 性能。

## 章节小结

本节给出了本章的能力目标、材料入口和学习顺序。下一节开始进入具体知识与实验。

## 实验工程说明与本章任务

本章实验工程位于 `src/mpi_spmv/`。`csr.cpp` 负责 CSR；`spmv_cpu.cpp`/`cpu_main.cpp` 是基线；`spmv_mpi.cpp` 完成分区、Scatterv、Bcast、local SpMV、Gatherv；`benchmark.cpp` 统计阶段时间；脚本负责构建和运行。

### 本章实验任务

构建 → 跑 CPU/1 rank 基线 → 用 2/4 rank 运行 → 检查累计 nnz 分区和 collective → 记录 compute/communication/total/error → 分析扩展性。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“Ranks、Total、Compute、Communication、Balance、Error”记录证据。
